# 05 · Análise de sobrevida

Análise de **tempo até um evento**, com censura à direita: para os países que ainda não tiveram o evento até 2100, sabemos apenas que o evento não ocorreu até lá. Ignorar esses países enviesaria o resultado para os que chegaram antes.

Dois eventos:
1. **Reposição:** primeiro ano em que a fecundidade fica abaixo de 2,1 filhos por mulher.
2. **Pico populacional:** ano em que a população atinge o máximo entre 1950 e 2100.

O tempo é medido em anos desde 1950. Os eventos vêm de **projeções** para a maioria dos países, então os resultados descrevem o cenário Medium, não observações.

**Cuidado com o tempo zero:** 4 países já estavam abaixo de 2,1 em 1950, então o tempo até o evento é zero. Eles são excluídos da análise da reposição.

**Covariáveis e grupos:** usam os valores de **1950**, para que a informação seja anterior ao evento. Os grupos vêm da clusterização transversal de 1950 (notebook 04).

**Dependências:** `lifelines`.

In [1]:
import sys
from pathlib import Path

# Permite importar src/ a partir da pasta notebooks/
if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import altair as alt
import numpy as np
import pandas as pd
from lifelines import CoxPHFitter, KaplanMeierFitter
from lifelines.statistics import multivariate_logrank_test
from lifelines.statistics import proportional_hazard_test

from src.dados import carregar_medium, paises

RAIZ = Path.cwd().parent
medium = carregar_medium()
paises_medium = paises(medium)
ORIGEM = 1950
FIM = 2100

EN_GRUPO = {
    "Baixa fecundidade": "Low fertility",
    "Transição avançada": "Advanced transition",
    "Transição inicial": "Early transition",
    "Transição intermediária": "Intermediate transition",
    "Alta fecundidade": "High fertility",
}

grupos_1950 = pd.read_csv(RAIZ / "data" / "processed" / "grupos_por_epoca.csv")
grupos_1950 = grupos_1950[grupos_1950["epoca"] == 1950][["LocID", "nome"]]
grupos_1950["nome"] = grupos_1950["nome"].map(EN_GRUPO)

# --- Figuras: fundo branco, texto preto, salvas em figures/ ---
import re as _re
import vl_convert as vlc
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_FIG = RAIZ / "figures"
PASTA_FIG.mkdir(exist_ok=True)
PREFIXO_FIG = "05"
_renderer_padrao = alt.renderers.get()
_contador_fig = {"n": 0}

def _titulo_de(spec):
    t = spec.get("title")
    if isinstance(t, dict):
        t = t.get("text")
    if isinstance(t, list):
        t = " ".join(map(str, t))
    if isinstance(t, str) and t:
        return t
    for chave in ("layer", "hconcat", "vconcat", "concat"):
        for sub in spec.get(chave, []):
            r = _titulo_de(sub)
            if r:
                return r
    return None

def _nome_grafico(spec):
    t = _titulo_de(spec) or "chart"
    slug = _re.sub(r"[^a-zA-Z0-9]+", "_", str(t).lower()).strip("_")
    return slug[:60] or "chart"

def _renderer_com_figuras(spec, **metadata):
    _contador_fig["n"] += 1
    base = f"{PREFIXO_FIG}_{_contador_fig['n']:02d}_{_nome_grafico(spec)}"
    spec_branco = dict(spec, background="white")
    (PASTA_FIG / f"{base}.png").write_bytes(vlc.vegalite_to_png(spec_branco, scale=2))
    if '"params"' in str(spec) and '"bind"' in str(spec):
        alt.VegaLite(spec_branco).save(str(PASTA_FIG / f"{base}.html"))
    return _renderer_padrao(spec, **metadata)

alt.renderers.register("com_figuras", _renderer_com_figuras)
alt.renderers.enable("com_figuras")



RendererRegistry.enable('com_figuras')

## Fundamentos

- **Função de sobrevivência S(t):** probabilidade de o evento ainda não ter ocorrido até o tempo t. Aqui o evento é a reposição ou o pico populacional.
- **Censura à direita:** o país que não teve o evento até 2100 entra na análise como "sobreviveu até 2100", sem data do evento. Descartá-lo subestimaria o tempo até o evento.
- **Kaplan-Meier:** estimador não paramétrico de S(t) que usa a censura corretamente.
- **Log-rank:** testa se as curvas de dois ou mais grupos são iguais em todo o período.
- **Modelo de Cox:** modela a taxa instantânea de ocorrência do evento (o risco) em função das covariáveis. A razão de risco (*hazard ratio*) acima de 1 indica aceleração; abaixo de 1, atraso.
- **Riscos proporcionais:** premissa do Cox de que o efeito de cada covariável é constante no tempo. Quando ela falha, usamos um termo que deixa o efeito variar com o tempo.

A pergunta é se as condições de 1950 (fecundidade e expectativa de vida) antecipam a chegada da reposição, como a teoria da transição sugere.

## 1. Construção das variáveis de tempo

Para cada país, o **tempo** é o número de anos entre 1950 e o evento. Se o evento não ocorreu até 2100, o tempo é censurado em 2100 (`evento = 0`).

In [2]:
base = paises_medium[paises_medium["Time"].between(ORIGEM, FIM)]
ini = base[base["Time"] == ORIGEM].set_index("LocID")[["Location", "TFR", "LEx"]].rename(
    columns={"TFR": "TFR_1950", "LEx": "LEx_1950"}
)

# Evento 1: reposição (TFR < 2,1 pela primeira vez)
primeiro_abaixo = base[base["TFR"] < 2.1].groupby("LocID")["Time"].min()
rep = ini.copy()
rep["ano_evento"] = primeiro_abaixo.reindex(rep.index)
rep["tempo"] = rep["ano_evento"].fillna(FIM) - ORIGEM
rep["evento"] = rep["ano_evento"].notna().astype(int)

ja_abaixo = rep[rep["tempo"] == 0]
print("Excluídos da reposição (já abaixo de 2,1 em 1950):", ", ".join(ja_abaixo["Location"]))
rep = rep[rep["tempo"] > 0]

# Evento 2: pico populacional (máximo entre 1950 e 2100)
idx_pico = base.groupby("LocID")["TPopulation1July"].idxmax()
pico = ini.copy()
pico["ano_evento"] = base.loc[idx_pico, ["LocID", "Time"]].set_index("LocID")["Time"]
pico["tempo"] = pico["ano_evento"] - ORIGEM
pico["evento"] = (pico["ano_evento"] < FIM).astype(int)
pico = pico[pico["tempo"] > 0]

print(f"Reposição: {len(rep)} países, {rep['evento'].sum()} eventos, {(rep['evento'] == 0).sum()} censurados")
print(f"Pico: {len(pico)} países, {pico['evento'].sum()} eventos, {(pico['evento'] == 0).sum()} censurados")

Excluídos da reposição (já abaixo de 2,1 em 1950): Latvia, Holy See, Austria, Luxembourg
Reposição: 233 países, 220 eventos, 13 censurados
Pico: 236 países, 161 eventos, 75 censurados


## 2. Curvas de Kaplan-Meier por grupo de 1950

A curva mostra a proporção de países que ainda **não** tiveram o evento em cada ano. A mediana é o ano em que metade dos países já teve o evento.

In [3]:
def curvas_km(df, coluna_grupo, nome_evento):
    linhas = []
    medianas = []
    for grupo, sub in df.dropna(subset=[coluna_grupo]).groupby(coluna_grupo):
        kmf = KaplanMeierFitter()
        kmf.fit(sub["tempo"], sub["evento"], label=grupo)
        sf = kmf.survival_function_.reset_index()
        sf.columns = ["tempo", "sobrevivencia"]
        sf["grupo"] = grupo
        sf["ano"] = ORIGEM + sf["tempo"]
        linhas.append(sf)
        mediana = kmf.median_survival_time_
        medianas.append({"grupo": grupo, "n": len(sub), "mediana (ano)":
                         ORIGEM + mediana if np.isfinite(mediana) else "não atingida"})
    curvas = pd.concat(linhas, ignore_index=True)
    grafico = (
        alt.Chart(curvas)
        .mark_line(interpolate="step-after", strokeWidth=2.5)
        .encode(
            x=alt.X("ano:Q", title="Year", axis=alt.Axis(format="d"), scale=alt.Scale(zero=False)),
            y=alt.Y("sobrevivencia:Q", title=f"Share without {nome_evento}", scale=alt.Scale(domain=[0, 1])),
            color=alt.Color("grupo:N", title="Group in 1950"),
            tooltip=["grupo:N", alt.Tooltip("ano:Q", format="d"), alt.Tooltip("sobrevivencia:Q", format=".2f")],
        )
        .properties(width=620, height=340)
    )
    return grafico, pd.DataFrame(medianas)

grafico_rep, medianas_rep = curvas_km(rep.join(grupos_1950.set_index("LocID")), "nome", "replacement")
display(medianas_rep.round(0))
grafico_rep

,grupo,n,mediana (ano)
0,Advanced transition,33,1973.0
1,Early transition,62,1992.0
2,High fertility,138,2044.0


alt.Chart(...)

**Leitura das medianas:** nenhum país era de baixa fecundidade em 1950, então esse grupo não aparece. Os de transição avançada atingem a reposição em torno de 1973, os de transição inicial em 1992 e os de alta fecundidade em 2044. A diferença é grande: os de alta fecundidade levam cerca de 70 anos a mais que os de transição avançada.

**Teste log-rank:** verifica se as curvas dos grupos diferem entre si. Ele compara todos os anos ao mesmo tempo e não supõe a forma das curvas. Um p-valor baixo indica que pelo menos um grupo tem um ritmo diferente dos outros.

In [4]:
com_grupo = rep.join(grupos_1950.set_index("LocID")).dropna(subset=["nome"])
teste = multivariate_logrank_test(com_grupo["tempo"], com_grupo["nome"], com_grupo["evento"])
print(f"Log-rank multigrupo: estatística = {teste.test_statistic:.1f}, p = {teste.p_value:.2g}")

Log-rank multigrupo: estatística = 229.0, p = 1.8e-50


**Resultado do teste log-rank:** há diferença entre os grupos (estatística de 229, p < 0,001).

## 3. Modelo de Cox: quais características antecipam a reposição?

O modelo de Cox estima a **razão de risco** (*hazard ratio*) de cada covariável: quanto ela acelera (acima de 1) ou atrasa (abaixo de 1) a chegada da reposição. As covariáveis estão padronizadas, então o efeito é por desvio-padrão.

In [5]:
cox_df = rep[["tempo", "evento", "TFR_1950", "LEx_1950"]].copy()
for c in ["TFR_1950", "LEx_1950"]:
    cox_df[c] = (cox_df[c] - cox_df[c].mean()) / cox_df[c].std()

cox = CoxPHFitter()
cox.fit(cox_df, duration_col="tempo", event_col="evento")
cox.print_summary(decimals=3, columns=["coef", "exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"])

<lifelines.CoxPHFitter: fitted with 233 total observations, 13 right-censored observations>
             duration col = 'tempo'
                event col = 'evento'
      baseline estimation = breslow
   number of observations = 233
number of events observed = 220
   partial log-likelihood = -937.033
         time fit was run = 2026-10-30 22:20:54 UTC

---
            coef exp(coef) exp(coef) lower 95% exp(coef) upper 95%       p
covariate                                                                 
TFR_1950  -0.678     0.508               0.409               0.631 <0.0005
LEx_1950   0.576     1.778               1.459               2.167 <0.0005
---
Concordance = 0.772
Partial AIC = 1878.067
log-likelihood ratio test = 162.302 on 2 df
-log2(p) of ll-ratio test = 117.076

**Leitura do Cox:**
- **Fecundidade em 1950:** razão de risco de 0,51 por desvio-padrão (IC 95% de 0,41 a 0,63). Países que começaram com mais filhos por mulher demoraram mais para chegar a 2,1.
- **Expectativa de vida em 1950:** razão de risco de 1,78 (IC 95% de 1,46 a 2,17). Quanto maior a expectativa de vida inicial, mais cedo a reposição, o que é coerente com a transição demográfica.
- **Concordância de 0,77:** o modelo ordena bem os países por tempo até o evento.


**Suposição de riscos proporcionais:** o Cox assume que o efeito de cada covariável é constante no tempo. O teste de Schoenfeld verifica isso. Um p-valor baixo sugere que o efeito muda ao longo do tempo, e a interpretação deve considerar o tempo (seção 4).

In [6]:
prop = proportional_hazard_test(cox, cox_df, time_transform="rank")
prop.summary.round(3)

,test_statistic,p,-log2(p)
LEx_1950,0.084,0.771,0.374
TFR_1950,22.218,0.000,18.649


**Atenção:** o efeito da fecundidade de 1950 não é constante no tempo (ver teste acima). Isso significa que a razão de risco de 0,51 é uma média: ela esconde um efeito que muda entre as épocas. A seção 4 trata isso com um termo que depende do tempo.

## 4. Efeito da fecundidade que muda no tempo

O teste de riscos proporcionais falhou para a fecundidade de 1950. A forma usual de tratar isso é deixar o coeficiente depender do tempo: incluímos o termo TFR × log(tempo). A razão de risco passa a ser uma função do horizonte, e não um número único.

Uma estratificação por época não serve aqui: o tempo de cada país é contado desde 1950, e cortar por época mudaria a própria pergunta do modelo.

In [7]:
cox_tv_df = cox_df.copy()
cox_tv_df["TFR_x_log_tempo"] = cox_tv_df["TFR_1950"] * np.log(cox_tv_df["tempo"])

cox_tv = CoxPHFitter()
cox_tv.fit(cox_tv_df, duration_col="tempo", event_col="evento")

coef_tv = cox_tv.params_
razao_por_horizonte = pd.DataFrame([
    {"anos desde 1950": h, "ano": ORIGEM + h,
     "razão de risco (TFR de 1950, por DP)": round(float(np.exp(coef_tv["TFR_1950"] + coef_tv["TFR_x_log_tempo"] * np.log(h))), 3)}
    for h in [10, 30, 60, 100, 150]
])
print("Log-verossimilhança:", round(cox_tv.log_likelihood_, 1), "| concordância:", round(cox_tv.concordance_index_, 3))
razao_por_horizonte

Log-verossimilhança: -909.0 | concordância: 0.718


,anos desde 1950,ano,"razão de risco (TFR de 1950, por DP)"
0,10,1960,0.077
1,30,1980,0.227
2,60,2010,0.449
3,100,2050,0.743
4,150,2100,1.106


**Leitura:** a razão de risco da fecundidade de 1950 muda de sinal ao longo do tempo. Nos primeiros 30 anos, cada desvio-padrão a mais de fecundidade em 1950 reduz fortemente a chance de chegar a 2,1 (razão de 0,08 em 10 anos e 0,23 em 30). Por volta de 100 anos, a razão de risco se aproxima de 1, que significa efeito nulo (0,74 em 2050). No horizonte de 150 anos o sinal se inverte (razão de 1,11): nesse horizonte, países que começaram com fecundidade alta chegam à reposição mais cedo do que os de fecundidade baixa.

A verossimilhança melhora com o termo de tempo (−909 contra −937), e a concordância cai de 0,77 para 0,72. A queda da concordância é esperada: o índice compara pares de países em horizontes fixos, e o efeito agora muda com o horizonte. Para o risco de cada país, o efeito por horizonte é o que importa.

## 5. Pico populacional

O pico é censurado quando a população ainda cresce em 2100. Os países que ainda crescem são o principal grupo censurado.

In [8]:
pico_grupos = pico.join(grupos_1950.set_index("LocID"))
grafico_pico, medianas_pico = curvas_km(pico_grupos, "nome", "population peak")
display(medianas_pico.round(0))
grafico_pico

,grupo,n,mediana (ano)
0,Advanced transition,37,2042.0
1,Early transition,62,2023.0
2,High fertility,137,2090.0


alt.Chart(...)

**Leitura do pico populacional:** 75 países ainda crescem em 2100 e ficam censurados. As medianas são 2023 (transição inicial), 2042 (transição avançada) e 2090 (alta fecundidade). O pico dos países que começaram com alta fecundidade fica para o fim do século.

## 6. Salvar resultados

- `figures/sobrevida_reposicao.html`: curvas de Kaplan-Meier da reposição (interativa).
- `figures/sobrevida_pico.html`: curvas do pico populacional.

In [9]:
SAIDA = RAIZ / "figures"
SAIDA.mkdir(exist_ok=True)
grafico_rep.configure(background="white").save(str(SAIDA / "sobrevida_reposicao.html"))
grafico_pico.configure(background="white").save(str(SAIDA / "sobrevida_pico.html"))
print("Salvos em", SAIDA)

Salvos em /Users/marcosabilio/Documents/GitHub/DataQuest2026/figures
